# Scheduling & Webhooks

This notebook covers two ways to run and observe workflows without an operator sitting at the keyboard:

1. **Schedules** — ask the server to run a workflow at a future UTC time (`client.schedules`).
2. **Webhooks** — subscribe an HTTP endpoint to workflow lifecycle events, inspect the delivery
   history, retry failures, and verify inbound signatures (`client.webhooks`).

Everything here is real, runnable SDK code. It creates a throwaway workflow, a schedule, and a
webhook subscription, then deletes all of them in the final cleanup cell.

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

from datetime import datetime, timedelta, timezone

import nest_asyncio

from interactly import AsyncWorkflowClient
from interactly.types.schedules.schedule import ScheduleStatus
from interactly.types.webhooks.webhook import WebhookAction, WebhookEventStatus

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## 1. A workflow to schedule

Schedules attach to a workflow, so create one first. We keep it deliberately tiny — a **single
`SayStaticMessageNodeConfig` start node that emits one fixed sentence and ends**. Because the
output is a known, fixed line, once the schedule fires we can look at the run and confirm the
scheduled workflow actually executed.

In [ ]:
from interactly.configs import (
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)
from interactly.types.workflows.workflow import Workflow

# A single StaticSay start node that emits one fixed sentence and ends.
# When the schedule fires, this exact line is what the run produces — so we
# can look at the run afterwards and confirm the scheduled workflow ran.
SCHEDULED_MESSAGE = "This scheduled follow-up ran successfully. Passed-in customer name is: {{customer_name}}"

say_node = SayStaticMessageNodeConfig(
    name="Scheduled Greeting",
    is_start=True,
    wait_for_user_message=False,
    static_messages_config=StaticMessagesConfig(static_messages=[SCHEDULED_MESSAGE]),
)

config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(
        name="Scheduled Follow-up (06_scheduling_and_webhooks)",
        description="Single StaticSay node used by the scheduling & webhooks notebook.",
    ),
    node_configs=[say_node],
    edge_configs=[],
)

workflow: Workflow = await client.workflows.create_from_config(config)
WF_ID = workflow.id
print(f"Workflow id={WF_ID}  name={workflow.name!r}")
print(f"When scheduled, it will emit: {SCHEDULED_MESSAGE!r}")

## 2. Schedule a run

`schedules.create()` takes a **future UTC `datetime`**. Optionally pass a `run_input` payload, a
specific workflow `version`, and a `scheduled_by_name` for the audit UI.

In [ ]:
# Schedule 5 minutes out, so you can wait a few minutes and confirm the run actually fired.
scheduled_time = datetime.now(timezone.utc) + timedelta(minutes=5)

schedule = await client.schedules.create(
    WF_ID,
    scheduled_time=scheduled_time,
    run_input={"dynamic_variables": {"customer_name": "Ada"}},
    scheduled_by_name="notebook-demo",
)
SCHEDULE_ID = schedule.id
print(f"Schedule id={SCHEDULE_ID}  status={schedule.status}  at={schedule.scheduled_time}")

## 3. List, filter, and fetch schedules

`list()` is scoped to one workflow; `list_all()` spans the whole team. Both use offset paging
(`skip`/`limit`) and accept a `ScheduleStatus` filter.

In [ ]:
per_workflow = await client.schedules.list(WF_ID, status=ScheduleStatus.PENDING, limit=10)
print(f"{len(per_workflow)} pending schedule(s) on this workflow")

team_wide = await client.schedules.list_all(limit=5)
print(f"{len(team_wide)} schedule(s) across the team (first page)")

fetched = await client.schedules.get(SCHEDULE_ID)
print(f"Re-fetched schedule status={fetched.status}")

## 4. Update, then cancel a pending schedule

Only `PENDING` schedules can be modified. To demonstrate `update()` and `cancel()` **without
touching the real 5-minute schedule** from section 2, we spin up a *separate throwaway schedule*
here, move its time, then cancel it. Moving `scheduled_time` re-creates the underlying EventBridge
schedule server-side; `cancel()` deletes it and returns `None`. The real schedule stays `PENDING`
so it can still fire.

In [ ]:
# Spin up a *separate* throwaway schedule just to demo update + cancel,
# so the real 5-minute schedule (SCHEDULE_ID) is left untouched and can fire.
demo_schedule = await client.schedules.create(
    WF_ID,
    scheduled_time=datetime.now(timezone.utc) + timedelta(hours=1),
    scheduled_by_name="notebook-demo (throwaway)",
)
print(f"Created throwaway schedule id={demo_schedule.id}  at={demo_schedule.scheduled_time}")

new_time = datetime.now(timezone.utc) + timedelta(hours=2)
updated = await client.schedules.update(demo_schedule.id, scheduled_time=new_time)
print(f"Rescheduled throwaway to {updated.scheduled_time}")

await client.schedules.cancel(demo_schedule.id)
print("Throwaway schedule cancelled.")
print(f"Real schedule {SCHEDULE_ID} is still pending and will fire at its scheduled time.")

## 5. Create a webhook subscription

A subscription points at an HTTPS endpoint and lists the `WebhookAction`s that trigger delivery.
Delivery behaviour (timeout, retries, backoff) is tunable. An optional `bearer_token` is sent in
the `Authorization` header; the returned model only reports `has_bearer_token` (never the value).

In [ ]:
subscription = await client.webhooks.create(
    name="Run lifecycle listener (notebook)",
    url="https://example.com/interactly/webhook",
    actions=[
        WebhookAction.WORKFLOW_RUN_STARTED,
        WebhookAction.WORKFLOW_RUN_COMPLETED,
    ],
    enabled=True,
    timeout_seconds=10,
    max_retries=3,
    retry_backoff_seconds=10,
)
SUB_ID = subscription.id
print(f"Subscription id={SUB_ID}  actions={[a.value for a in subscription.actions]}")

## 6. List, fetch, and update subscriptions

`list()` is paginated (`page`/`size`) and filterable by `search`, `enabled`, and `action`.
`get()` has no server route — the SDK resolves it client-side by walking the list.

In [ ]:
page = await client.webhooks.list(enabled=True, action=WebhookAction.WORKFLOW_RUN_COMPLETED)
print(f"{page.total} enabled subscription(s) matching the action filter")

one = await client.webhooks.get(SUB_ID)
print(f"Fetched {one.name!r}")

# Disable it and drop the retry count.
updated_sub = await client.webhooks.update(SUB_ID, enabled=False, max_retries=1)
print(f"enabled={updated_sub.enabled}  max_retries={updated_sub.max_retries}")

## 7. Inspect the delivery history

`list_events()` returns dispatched events (newest first); each `WebhookEvent` carries its status,
attempt count, and the last HTTP status/error. `list_delivery_attempts()` drills into a single
event's attempts. `retry_event()` re-dispatches a failed/pending event **by its document `id`**
(not the `event_id` string field). Guarded here because a fresh subscription has no events yet.

In [ ]:
events = await client.webhooks.list_events(subscription_id=SUB_ID, size=10)
print(f"{events.total} event(s) for this subscription")

first = next(iter(events.items), None)
if first is not None:
    print(f"event {first.event_id}  status={first.status}  attempts={first.attempts_count}")
    attempts = await client.webhooks.list_delivery_attempts(first.event_id, size=10)
    print(f"  {attempts.total} delivery attempt(s)")
    if first.status in (WebhookEventStatus.FAILED, WebhookEventStatus.PENDING) and first.id:
        result = await client.webhooks.retry_event(first.id)
        print(f"  retry -> {result}")
else:
    print("No events yet — trigger a matching workflow action to generate deliveries.")

## 8. Verify an inbound signature

Interactly signs every outbound POST with `HMAC-SHA256(secret, body)` and sends the hex digest in
the `X-Interactly-Signature` header. In your receiving endpoint, call `verify_signature()` on the
**raw** body. Below we sign a payload the same way the server would, then verify it — and show a
tampered payload being rejected. (`verify_signature` is a pure helper — no `await`.)

In [ ]:
import hashlib
import hmac

from interactly.webhooks import verify_signature, WebhookVerificationError

secret = "whsec_example_secret"
payload = b'{"action": "workflow_run_completed", "workflow_run_id": "run_123"}'
signature = hmac.new(secret.encode(), payload, hashlib.sha256).hexdigest()

verify_signature(payload=payload, secret=secret, signature_header=signature)
print("Valid signature accepted.")

try:
    verify_signature(payload=payload + b"tampered", secret=secret, signature_header=signature)
except WebhookVerificationError as exc:
    print(f"Tampered payload rejected: {exc}")

## 9. Cleanup

Remove the subscription, cancel the still-pending real schedule, and delete the workflow.

> **Want to watch the schedule fire first?** The real 5-minute schedule from section 2 is still
> `PENDING`. **Skip this cell**, wait until its scheduled time, and check the run for the
> `"This scheduled follow-up ran successfully."` line — then come back and run cleanup. Running it
> now cancels the schedule before it can fire. (The `cancel()` below is wrapped in a `try` so it
> won't error if the schedule already fired.)

In [ ]:
await client.webhooks.delete(SUB_ID)

# Cancel the real 5-minute schedule if it is still pending. If it has already
# fired (or was cancelled), cancel() will raise — we swallow that so cleanup
# is safe to run at any time.
try:
    await client.schedules.cancel(SCHEDULE_ID)
    print("Cancelled the pending schedule.")
except Exception as exc:
    print(f"Schedule not cancelled (already fired or gone): {exc}")

await client.workflows.delete(WF_ID)
print("Deleted webhook subscription and workflow.")

## See also

- Guides: [`../docs/guides/scheduling.md`](../docs/guides/scheduling.md) and [`../docs/guides/webhooks.md`](../docs/guides/webhooks.md)
- [`04_pagination_and_filtering.ipynb`](04_pagination_and_filtering.ipynb) — paging over `list_events()` and subscriptions
- [`05_error_handling.ipynb`](05_error_handling.ipynb) — catching `NotFoundError` from `webhooks.get()` / `retry_event()`
- [`03_streaming_events.ipynb`](03_streaming_events.ipynb) — the in-process alternative to outbound webhooks